In [ ]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


In [ ]:
train = pd.read_csv('/kaggle/input/datasets/harshilgoel2007/trafficdemad/train.csv')
given_test = pd.read_csv('/kaggle/input/datasets/harshilgoel2007/trafficdemad/test.csv')
test = given_test.copy()

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.nunique()

In [ ]:
test.head()

In [ ]:
test.nunique()

In [ ]:
!pip install pygeohash

In [ ]:
import pygeohash as pgh
def transform_data(df):

    df = df.copy()
    
    df["Temperature"] = df["Temperature"].fillna(df["Temperature"].median())
    df["RoadType"] = df["RoadType"].fillna(df["RoadType"].mode()[0])
    df["Weather"] = df["Weather"].fillna(df["Weather"].mode()[0])

    df["hour"] = df["timestamp"].str.split(":").str[0].astype(int)
    df["minute"] = df["timestamp"].str.split(":").str[1].astype(int)

    df["time_in_minutes"] = df["hour"] * 60 + df["minute"]

    df["is_morning_peak"] = df["hour"].between(7,10).astype(int)
    df["is_evening_peak"] = df["hour"].between(16,20).astype(int)

    df["is_weekend_like"] = (df["day"] == 49).astype(int)

    df = pd.get_dummies(df,columns=['RoadType','Weather'])
    df['LargeVehicles'] = (df['LargeVehicles'] == 'Allowed').astype(int)
    df['Landmarks'] = (df['Landmarks'] == 'Yes').astype(int)
    df['lat'] = df['geohash'].apply(lambda x: pgh.decode(x)[0])
    df['lon'] = df['geohash'].apply(lambda x: pgh.decode(x)[1])
    df = df.drop(columns=['timestamp','Index','geohash','day'])

    return df

In [ ]:
test = transform_data(test)
train = transform_data(train)

In [ ]:
test.head()

In [ ]:
train.head()

In [ ]:
X = train.drop(columns=["demand"])
y = train["demand"]


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score
from catboost import CatBoostRegressor

model = CatBoostRegressor(
    iterations=7000,
    depth=7,
    learning_rate=0.03,
    loss_function="RMSE",
    eval_metric="R2",
    random_seed=42,
    verbose=250
)

model.fit(
    X_train,
    y_train,
    eval_set=(X_valid, y_valid),
    use_best_model=True
)

pred = model.predict(X_valid)

actual = y_valid

print("Validation R2 =", r2_score(actual, pred))

In [ ]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

plt.figure(figsize=(10,8))
sns.barplot(data=importance.head(15), x="Importance", y="Feature")
plt.title("Top Features")
plt.show()

importance.head(20)

In [ ]:
test_preds = model.predict(test)

submission = pd.DataFrame({
    "Index": given_test["Index"],
    "demand": test_preds
})

submission.to_csv("sample_submission.csv", index=False)

print("sample_submission.csv saved")
print(submission.shape)
submission.head()